# FuXi Checkpoint Comparison Study

This notebook performs a full, reproducible comparison of two checkpoints:
- emb_768: `/home/raj.ayush/fuxi-final/fuxi_new/Models_paper/pretrain/emb_768/best.pt`
- emb_1536: `/home/raj.ayush/fuxi-final/fuxi_new/Models_paper/pretrain/emb_1536/best.pt`

Metrics:
- Latitude-weighted RMSE
- ACC (Anomaly Correlation Coefficient)

The workflow includes:
1. Environment setup
2. Config and reproducibility
3. Data input
4. Validation and profiling
5. Core evaluation logic
6. Visual analysis
7. Export of artifacts
8. Quick verification

Run cells from top to bottom.

## 1) Environment Setup and Package Imports

This section imports all required packages and applies plotting defaults.

If any package is missing, uncomment and run the install line in the next cell.

In [ ]:
# Optional install (uncomment if needed):
# !pip install numpy pandas matplotlib seaborn torch zarr xarray

from __future__ import annotations

import gc
import json
import math
import random
import warnings
from contextlib import nullcontext
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from torch.utils.data import DataLoader, Dataset
import xarray as xr
import zarr

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")

print("Imports loaded successfully.")
print("PyTorch:", torch.__version__)

## 2) Project Configuration and Reproducibility

All configurable constants are defined in one dictionary for easy edits.

Important:
- Keep both checkpoints on the same data split.
- If you run this on `gpu2`, use `device_preference='cuda'`.

In [ ]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path('/home/raj.ayush/fuxi-final/fuxi_new')

CONFIG = {
    'project_root': PROJECT_ROOT,
    'checkpoint_paths': {
        'emb_768': PROJECT_ROOT / 'Models_paper/pretrain/emb_768/best.pt',
        'emb_1536': PROJECT_ROOT / 'Models_paper/pretrain/emb_1536/best.pt',
    },
    'main_data_store': Path('/home/bedartha/public/datasets/for_model_development/weatherbench2/era5/1979-2022_01_10-6h-240x121_equiangular_with_poles_conservative_MWE.zarr'),
    'climatology_store': PROJECT_ROOT / 'data/climatology/1990-2019_6h_240x121_equiangular_with_poles_conservative_MWE.zarr',
    'train_start': '1979-01-01',
    'train_end': '2019-12-31',
    'test_start': '2021-01-01',
    'test_end': '2022-12-31',
    'rollout_steps': 60,            # 60 x 6h = 15 days
    'history_steps': 2,
    'batch_size': 2,
    'num_workers': 2,
    'max_samples': None,            # set e.g. 256 for faster debug runs
    'stats_samples': 256,
    'device_preference': 'cuda',    # 'cuda', 'cpu', or 'auto'
    'amp': 'bf16',                  # 'none', 'fp16', 'bf16'
    'output_dir': PROJECT_ROOT / 'results/checkpoint_compare_emb768_emb1536',
}

CONFIG['output_dir'].mkdir(parents=True, exist_ok=True)

# Allow importing from src/
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print('Output directory:', CONFIG['output_dir'])

In [ ]:
def choose_device(mode: str) -> torch.device:
    if mode == 'cpu':
        return torch.device('cpu')
    if mode == 'cuda':
        if not torch.cuda.is_available():
            raise RuntimeError('CUDA requested but not available.')
        return torch.device('cuda:0')
    if torch.cuda.is_available():
        return torch.device('cuda:0')
    return torch.device('cpu')


def autocast_ctx(device: torch.device, amp: str):
    if device.type != 'cuda' or amp == 'none':
        return nullcontext()
    dtype = torch.bfloat16 if amp == 'bf16' else torch.float16
    return torch.autocast(device_type='cuda', dtype=dtype)


device = choose_device(CONFIG['device_preference'])
if device.type == 'cuda':
    torch.cuda.set_device(0)
    torch.backends.cudnn.benchmark = True

print('Device selected:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## 3) Data Input (File/URL/API)

This section loads:
- Checkpoint metadata (`config` from `.pt` files)
- WeatherBench2 Zarr data store
- Optional climatology store

It also prints quick previews (`head`, shape, dtypes) where relevant.

In [ ]:
def load_checkpoint_meta(path: Path) -> Dict:
    if not path.is_file():
        raise FileNotFoundError(f'Checkpoint not found: {path}')
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    cfg = ckpt.get('config', {})
    return {
        'path': path,
        'config': cfg,
        'checkpoint': ckpt,
    }


ckpt_bundle = {name: load_checkpoint_meta(path) for name, path in CONFIG['checkpoint_paths'].items()}

summary_rows = []
for name, bundle in ckpt_bundle.items():
    cfg = bundle['config']
    summary_rows.append(
        {
            'checkpoint_name': name,
            'embed_dim': cfg.get('embed_dim'),
            'num_heads': cfg.get('num_heads'),
            'window_size': cfg.get('window_size'),
            'depth_pre': cfg.get('depth_pre'),
            'depth_mid': cfg.get('depth_mid'),
            'depth_post': cfg.get('depth_post'),
            'train_start': cfg.get('train_start'),
            'train_end': cfg.get('train_end'),
            'test_start': cfg.get('test_start'),
            'test_end': cfg.get('test_end'),
        }
    )

ckpt_df = pd.DataFrame(summary_rows)
display(ckpt_df)

# Use the first checkpoint as reference for variable choices.
reference_cfg = next(iter(ckpt_bundle.values()))['config']
pressure_vars_requested = reference_cfg.get('pressure_vars', ['temperature', 'geopotential', 'specific_humidity', 'u_component_of_wind', 'v_component_of_wind'])
surface_vars_requested = reference_cfg.get('surface_vars', ['t2m', 'u10', 'v20', 'mslp', 'tcwv'])
pressure_levels_requested = reference_cfg.get('pressure_levels', [850, 500, 250])

In [ ]:
# Attempt opening the main data store and optional climatology store.
main_store_path = Path(reference_cfg.get('zarr_store', str(CONFIG['main_data_store'])))
climo_store_path = Path(CONFIG['climatology_store'])

print('Main data store:', main_store_path)
print('Climatology store:', climo_store_path)

main_store = zarr.open_group(str(main_store_path), mode='r')
print('Main store keys (sample):', sorted(list(main_store.keys()))[:10])

try:
    climo_store = zarr.open_group(str(climo_store_path), mode='r')
    climo_readable = True
    print('Climatology store readable. Keys (sample):', sorted(list(climo_store.keys()))[:10])
except Exception as e:
    climo_store = None
    climo_readable = False
    print('Climatology store not readable:', type(e).__name__, str(e)[:160])

## 4) Data Validation and Basic Profiling

This section validates:
- Required files exist
- Time range availability
- Variable mapping and channel count
- Basic dataset diagnostics

In [ ]:
PRESSURE_VAR_ALIASES: Dict[str, str] = {
    'temperature': 'temperature',
    't': 'temperature',
    'geopotential': 'geopotential',
    'z': 'geopotential',
    'specific_humidity': 'specific_humidity',
    'q': 'specific_humidity',
    'u_component_of_wind': 'u_component_of_wind',
    'u': 'u_component_of_wind',
    'v_component_of_wind': 'v_component_of_wind',
    'v': 'v_component_of_wind',
}

SURFACE_VAR_ALIASES: Dict[str, str] = {
    '2m_temperature': '2m_temperature',
    't2m': '2m_temperature',
    '10m_u_component_of_wind': '10m_u_component_of_wind',
    'u10': '10m_u_component_of_wind',
    '10m_v_component_of_wind': '10m_v_component_of_wind',
    'v10': '10m_v_component_of_wind',
    'v20': '10m_v_component_of_wind',
    'mean_sea_level_pressure': 'mean_sea_level_pressure',
    'mslp': 'mean_sea_level_pressure',
    'surface_pressure': 'surface_pressure',
    'sp': 'surface_pressure',
    'total_column_water_vapour': 'total_column_water_vapour',
    'tcwv': 'total_column_water_vapour',
}


def resolve_variable_names(zarr_path: str, requested_pressure_vars: Sequence[str], requested_surface_vars: Sequence[str]) -> Tuple[List[str], List[str], List[str]]:
    store = zarr.open_group(zarr_path, mode='r')
    available = set(store.keys())
    notes: List[str] = []

    def resolve_one(name: str, aliases: Dict[str, str], kind: str) -> str:
        key = name.strip()
        key_lower = key.lower()
        if key in available:
            return key
        mapped = aliases.get(key_lower, key)
        if mapped in available:
            if mapped != key:
                notes.append(f"{kind}: mapped '{key}' -> '{mapped}'")
            return mapped
        if key_lower in ('mslp', 'mean_sea_level_pressure') and 'surface_pressure' in available:
            notes.append(f"{kind}: '{key}' missing, using fallback 'surface_pressure'")
            return 'surface_pressure'
        raise KeyError(f"Could not resolve {kind} variable '{key}'.")

    resolved_pressure = [resolve_one(v, PRESSURE_VAR_ALIASES, 'pressure') for v in requested_pressure_vars]
    resolved_surface = [resolve_one(v, SURFACE_VAR_ALIASES, 'surface') for v in requested_surface_vars]
    return resolved_pressure, resolved_surface, notes


resolved_pressure_vars, resolved_surface_vars, resolution_notes = resolve_variable_names(
    str(main_store_path),
    pressure_vars_requested,
    surface_vars_requested,
)

print('Resolved pressure vars:', resolved_pressure_vars)
print('Resolved surface vars:', resolved_surface_vars)
print('Pressure levels:', pressure_levels_requested)
if resolution_notes:
    print('Resolution notes:')
    for note in resolution_notes:
        print(' -', note)

In [ ]:
# Quick profile of coordinates and time axis from the main store.
lat = np.asarray(main_store['latitude'][:]) if 'latitude' in main_store else np.asarray(main_store['lat'][:])
lon = np.asarray(main_store['longitude'][:]) if 'longitude' in main_store else np.asarray(main_store['lon'][:])
raw_time = np.asarray(main_store['time'][:])
time_units = dict(main_store['time'].attrs).get('units', 'hours since 1959-01-01')

print('Latitude shape:', lat.shape, 'dtype:', lat.dtype)
print('Longitude shape:', lon.shape, 'dtype:', lon.dtype)
print('Time shape:', raw_time.shape, 'dtype:', raw_time.dtype)
print('Time units:', time_units)

assert len(lat.shape) == 1 and len(lon.shape) == 1, 'Latitude/longitude must be 1D.'
assert len(resolved_pressure_vars) > 0 and len(resolved_surface_vars) > 0, 'Variable lists cannot be empty.'

schema_df = pd.DataFrame(
    {
        'group': ['pressure_vars', 'surface_vars', 'pressure_levels'],
        'count': [len(resolved_pressure_vars), len(resolved_surface_vars), len(pressure_levels_requested)],
        'values': [
            ', '.join(resolved_pressure_vars),
            ', '.join(resolved_surface_vars),
            ', '.join([str(v) for v in pressure_levels_requested]),
        ],
    }
)
display(schema_df)

## 5) Core Processing Logic

This is the main evaluation module in notebook form:
- Data accessor for WeatherBench2 Zarr
- Channel normalization stats
- Rollout dataset for autoregressive evaluation
- Climatology preparation
- Latitude-weighted RMSE and ACC computation
- Checkpoint model loading and batched evaluation

In [ ]:
from src.models.fuxi_model import FuXi


class WB2Accessor:
    def __init__(self, zarr_path: str, pressure_vars: Sequence[str], surface_vars: Sequence[str], pressure_levels: Sequence[int]):
        if not Path(zarr_path).exists():
            raise FileNotFoundError(f'Zarr store not found: {zarr_path}')

        self.store = zarr.open_group(zarr_path, mode='r')
        self.pressure_vars = list(pressure_vars)
        self.surface_vars = list(surface_vars)
        self.pressure_levels = [int(v) for v in pressure_levels]

        self.latitudes = self._get_coord('latitude', 'lat')
        self.longitudes = self._get_coord('longitude', 'lon')
        self.spatial_shape = (int(self.latitudes.shape[0]), int(self.longitudes.shape[0]))

        self.time_values = self._decode_time_axis()
        self._resolve_level_indices()
        self._bind_arrays()
        self._infer_spatial_order()

        self.var_names = [
            f"{v}_plev{p}" for v in self.pressure_vars for p in self.pressure_levels
        ] + list(self.surface_vars)
        self.channels = len(self.var_names)

    def _get_coord(self, primary: str, secondary: str) -> np.ndarray:
        if primary in self.store:
            return np.asarray(self.store[primary][:])
        if secondary in self.store:
            return np.asarray(self.store[secondary][:])
        raise KeyError(f"Missing coordinate: {primary}/{secondary}")

    def _decode_time_axis(self) -> np.ndarray:
        raw_time = np.asarray(self.store['time'][:])
        attrs = dict(self.store['time'].attrs)
        units = attrs.get('units', 'hours since 1959-01-01')
        if ' since ' not in units:
            raise ValueError(f'Unsupported time units: {units}')

        delta_unit_raw, base_date_raw = units.split(' since ', 1)
        delta_unit = delta_unit_raw.strip().lower().rstrip('s')
        unit_map = {'hour': 'h', 'minute': 'm', 'second': 's', 'day': 'D'}
        if delta_unit not in unit_map:
            raise ValueError(f'Unsupported delta unit: {delta_unit}')

        base_date = np.datetime64(base_date_raw.strip())
        return base_date + raw_time.astype(np.int64).astype(f"timedelta64[{unit_map[delta_unit]}]")

    def time_indices_between(self, start_date: str, end_date: str) -> np.ndarray:
        mask = (self.time_values >= np.datetime64(start_date)) & (self.time_values <= np.datetime64(end_date))
        return np.where(mask)[0]

    def _resolve_level_indices(self) -> None:
        levels = np.asarray(self.store['level'][:]).astype(int)
        idx_by_level = {int(v): i for i, v in enumerate(levels.tolist())}
        missing = [v for v in self.pressure_levels if v not in idx_by_level]
        if missing:
            raise ValueError(f'Missing pressure levels: {missing}. Available: {levels.tolist()}')
        self._level_indices = [int(idx_by_level[v]) for v in self.pressure_levels]

    def _bind_arrays(self) -> None:
        self._pressure_arrays = [self.store[name] for name in self.pressure_vars]
        self._surface_arrays = [self.store[name] for name in self.surface_vars]

    def _infer_spatial_order(self) -> None:
        lat_n, lon_n = self.spatial_shape
        pshape = self._pressure_arrays[0].shape
        if pshape[2] == lat_n and pshape[3] == lon_n:
            self._transpose_spatial = False
        elif pshape[2] == lon_n and pshape[3] == lat_n:
            self._transpose_spatial = True
        else:
            raise ValueError(f'Cannot infer spatial order from shape={pshape}, lat/lon={self.spatial_shape}')

    def load_frame(self, abs_time_idx: int) -> np.ndarray:
        parts: List[np.ndarray] = []
        for arr in self._pressure_arrays:
            block = np.asarray(arr[abs_time_idx, self._level_indices, :, :], dtype=np.float32)
            if self._transpose_spatial:
                block = np.swapaxes(block, -2, -1)
            parts.append(block)

        for arr in self._surface_arrays:
            block = np.asarray(arr[abs_time_idx, :, :], dtype=np.float32)
            if self._transpose_spatial:
                block = np.swapaxes(block, -2, -1)
            parts.append(block[np.newaxis, ...])

        return np.concatenate(parts, axis=0)


def compute_channel_stats(accessor: WB2Accessor, time_indices: np.ndarray, stats_samples: int = 256) -> Tuple[torch.Tensor, torch.Tensor]:
    n = max(1, min(int(stats_samples), int(time_indices.shape[0])))
    sampled_pos = np.linspace(0, time_indices.shape[0] - 1, num=n, dtype=np.int64)

    ch_sum = np.zeros((accessor.channels,), dtype=np.float64)
    ch_sq = np.zeros((accessor.channels,), dtype=np.float64)
    pixels = 0

    for pos in sampled_pos.tolist():
        frame = accessor.load_frame(int(time_indices[pos]))
        flat = frame.reshape(frame.shape[0], -1).astype(np.float64)
        ch_sum += flat.sum(axis=1)
        ch_sq += np.square(flat).sum(axis=1)
        pixels += flat.shape[1]

    mean = ch_sum / max(pixels, 1)
    var = ch_sq / max(pixels, 1) - np.square(mean)
    std = np.sqrt(np.maximum(var, 1e-6))

    mean_t = torch.from_numpy(mean.astype(np.float32)).view(-1, 1, 1)
    std_t = torch.from_numpy(std.astype(np.float32)).view(-1, 1, 1)
    return mean_t, std_t


class RolloutDataset(Dataset):
    def __init__(
        self,
        accessor: WB2Accessor,
        eval_time_indices: np.ndarray,
        rollout_steps: int,
        mean: torch.Tensor,
        std: torch.Tensor,
        history_steps: int = 2,
        max_samples: Optional[int] = None,
    ):
        super().__init__()
        if history_steps != 2:
            raise ValueError('FuXi rollout expects exactly 2 history steps.')

        self.accessor = accessor
        self.eval_time_indices = eval_time_indices
        self.rollout_steps = int(rollout_steps)
        self.mean = mean
        self.std = std
        self.history_steps = history_steps

        n = eval_time_indices.shape[0]
        usable = n - history_steps - self.rollout_steps + 1
        if usable <= 0:
            raise ValueError(f'Not enough eval timesteps ({n}) for history={history_steps}, rollout={rollout_steps}')

        self.starts = np.arange(usable, dtype=np.int64)
        if max_samples is not None:
            self.starts = self.starts[: int(max_samples)]

    def __len__(self) -> int:
        return int(self.starts.shape[0])

    def __getitem__(self, idx: int):
        s = int(self.starts[idx])
        abs_idx = self.eval_time_indices

        f0 = self.accessor.load_frame(int(abs_idx[s + 0]))
        f1 = self.accessor.load_frame(int(abs_idx[s + 1]))
        h0 = (torch.from_numpy(f0) - self.mean) / self.std
        h1 = (torch.from_numpy(f1) - self.mean) / self.std
        history = torch.stack([h0, h1], dim=1)  # (C,2,H,W)

        future_list = []
        for k in range(self.rollout_steps):
            fr = self.accessor.load_frame(int(abs_idx[s + self.history_steps + k]))
            fr_t = (torch.from_numpy(fr) - self.mean) / self.std
            future_list.append(fr_t)
        future = torch.stack(future_list, dim=0)  # (S,C,H,W)

        init_time = str(self.accessor.time_values[int(abs_idx[s + 1])])
        return history, future, init_time


def build_lat_weights(latitudes: np.ndarray, device: torch.device) -> torch.Tensor:
    lat = torch.tensor(latitudes, dtype=torch.float32, device=device)
    w = torch.cos(torch.deg2rad(lat))
    w = w / w.mean().clamp(min=1e-8)
    return w.view(1, 1, -1, 1)


print('Core data classes and helper functions loaded.')

In [ ]:
def build_static_climatology_from_external_store(
    climo_store_path: Path,
    pressure_vars: Sequence[str],
    surface_vars: Sequence[str],
    pressure_levels: Sequence[int],
) -> Optional[np.ndarray]:
    """
    Attempt to build a static (C,H,W) climatology by averaging day/hour dimensions
    from an hourly climatology zarr store. Returns None if unavailable.
    """
    if not climo_store_path.exists():
        return None

    try:
        ds = xr.open_zarr(str(climo_store_path), consolidated=True)
    except Exception:
        try:
            ds = xr.open_zarr(str(climo_store_path), consolidated=False)
        except Exception:
            return None

    # Determine common coordinate names.
    lat_name = 'latitude' if 'latitude' in ds.coords else ('lat' if 'lat' in ds.coords else None)
    lon_name = 'longitude' if 'longitude' in ds.coords else ('lon' if 'lon' in ds.coords else None)
    if lat_name is None or lon_name is None:
        return None

    # Candidate temporal dims to average across.
    temporal_dims = [d for d in ['dayofyear', 'day', 'hour'] if d in ds.dims]

    out_parts = []

    for var in pressure_vars:
        if var not in ds:
            return None
        da = ds[var]
        if 'level' not in da.dims:
            return None
        da = da.sel(level=list(pressure_levels))

        dims_to_mean = [d for d in temporal_dims if d in da.dims]
        if dims_to_mean:
            da = da.mean(dim=dims_to_mean)

        da = da.transpose('level', lat_name, lon_name)
        out_parts.append(da.values.astype(np.float32))

    for var in surface_vars:
        if var not in ds:
            return None
        da = ds[var]

        dims_to_mean = [d for d in temporal_dims if d in da.dims]
        if dims_to_mean:
            da = da.mean(dim=dims_to_mean)

        da = da.transpose(lat_name, lon_name)
        out_parts.append(da.values.astype(np.float32)[np.newaxis, ...])

    if len(out_parts) == 0:
        return None

    clim = np.concatenate(out_parts, axis=0)
    return clim.astype(np.float32)


def build_static_climatology_from_train_period(
    accessor: WB2Accessor,
    train_indices: np.ndarray,
    sample_cap: Optional[int] = None,
) -> np.ndarray:
    idx = train_indices
    if sample_cap is not None and sample_cap < len(idx):
        select_pos = np.linspace(0, len(idx) - 1, sample_cap, dtype=np.int64)
        idx = idx[select_pos]

    ch_sum = np.zeros((accessor.channels, accessor.spatial_shape[0], accessor.spatial_shape[1]), dtype=np.float64)
    n = 0
    for t_idx in idx.tolist():
        frame = accessor.load_frame(int(t_idx)).astype(np.float64)
        ch_sum += frame
        n += 1

    return (ch_sum / max(n, 1)).astype(np.float32)


def build_model_from_checkpoint(ckpt: Dict, channels: int, spatial_shape: Tuple[int, int]) -> FuXi:
    cfg = ckpt.get('config', {})
    model = FuXi(
        num_variables=channels,
        embed_dim=int(cfg.get('embed_dim', 1536)),
        num_heads=int(cfg.get('num_heads', 8)),
        window_size=int(cfg.get('window_size', 8)),
        depth_pre=int(cfg.get('depth_pre', 2)),
        depth_mid=int(cfg.get('depth_mid', 44)),
        depth_post=int(cfg.get('depth_post', 2)),
        mlp_ratio=float(cfg.get('mlp_ratio', 4.0)),
        drop_path_rate=float(cfg.get('drop_path_rate', 0.2)),
        input_height=int(spatial_shape[0]),
        input_width=int(spatial_shape[1]),
        use_checkpoint=False,
    )
    model.load_state_dict(ckpt['model_state'], strict=True)
    model.eval()
    return model


@torch.no_grad()
def evaluate_rollout(
    model: FuXi,
    loader: DataLoader,
    mean: torch.Tensor,
    std: torch.Tensor,
    climatology: torch.Tensor,
    lat_w: torch.Tensor,
    device: torch.device,
    amp: str,
) -> Tuple[np.ndarray, np.ndarray]:
    rollout_steps = loader.dataset.rollout_steps
    channels = int(mean.shape[0])

    rmse_mse_sum = torch.zeros((rollout_steps, channels), dtype=torch.float64, device=device)
    acc_sum = torch.zeros((rollout_steps, channels), dtype=torch.float64, device=device)
    count = torch.zeros((rollout_steps,), dtype=torch.float64, device=device)

    mean = mean.to(device)
    std = std.to(device)
    clim = climatology.to(device)

    total_batches = len(loader)
    for batch_idx, (history, future, _init_time) in enumerate(loader, start=1):
        history = history.to(device, non_blocking=True)  # (B,C,2,H,W)
        future = future.to(device, non_blocking=True)    # (B,S,C,H,W)
        batch_size = history.shape[0]
        hist = history

        for s in range(rollout_steps):
            with autocast_ctx(device, amp):
                pred_n = model(hist)  # normalized prediction

            tgt_n = future[:, s]
            pred = pred_n.float() * std + mean
            tgt = tgt_n.float() * std + mean

            err = pred - tgt
            mse = (err.square() * lat_w).mean(dim=(2, 3))
            rmse_mse_sum[s] += mse.double().sum(dim=0)

            pred_anom = pred - clim
            tgt_anom = tgt - clim
            num = (pred_anom * tgt_anom * lat_w).sum(dim=(2, 3))
            den = torch.sqrt(
                (pred_anom.square() * lat_w).sum(dim=(2, 3))
                * (tgt_anom.square() * lat_w).sum(dim=(2, 3))
                + 1e-12
            )
            acc = num / den
            acc_sum[s] += acc.double().sum(dim=0)

            count[s] += float(batch_size)
            hist = torch.stack([hist[:, :, 1], pred_n], dim=2)

        if batch_idx == 1 or batch_idx % 10 == 0 or batch_idx == total_batches:
            step1_rmse = float(torch.sqrt((rmse_mse_sum[0] / count[0].clamp(min=1.0)).clamp(min=1e-12)).mean().item())
            step1_acc = float((acc_sum[0] / count[0].clamp(min=1.0)).mean().item())
            print(f"Batch {batch_idx}/{total_batches} | step1 mean RMSE={step1_rmse:.4f}, ACC={step1_acc:.4f}")

    denom = count.clamp(min=1.0).unsqueeze(1)
    rmse = torch.sqrt((rmse_mse_sum / denom).clamp(min=1e-12)).cpu().numpy()
    acc = (acc_sum / denom).cpu().numpy()
    return rmse, acc


print('Climatology, model, and rollout metric functions loaded.')

In [ ]:
# Build accessor and split indices.
accessor = WB2Accessor(
    zarr_path=str(main_store_path),
    pressure_vars=resolved_pressure_vars,
    surface_vars=resolved_surface_vars,
    pressure_levels=pressure_levels_requested,
)

train_idx = accessor.time_indices_between(CONFIG['train_start'], CONFIG['train_end'])
eval_idx = accessor.time_indices_between(CONFIG['test_start'], CONFIG['test_end'])

print('Train timesteps:', len(train_idx))
print('Eval timesteps:', len(eval_idx))
print('Channels:', accessor.channels)
print('Spatial shape:', accessor.spatial_shape)

# Compute normalization stats once (shared by both checkpoints).
mean, std = compute_channel_stats(accessor, train_idx, stats_samples=CONFIG['stats_samples'])

# Build static climatology: preferred source = user-provided climatology store.
clim_np = build_static_climatology_from_external_store(
    climo_store_path=climo_store_path,
    pressure_vars=resolved_pressure_vars,
    surface_vars=resolved_surface_vars,
    pressure_levels=pressure_levels_requested,
)

if clim_np is None:
    print('Falling back to train-period climatology from main store.')
    clim_np = build_static_climatology_from_train_period(
        accessor=accessor,
        train_indices=train_idx,
        sample_cap=None,
    )
else:
    print('Using external climatology store for static climatology map.')

climatology = torch.from_numpy(clim_np)
lat_w = build_lat_weights(accessor.latitudes, device)

# Create rollout dataset/loader (shared by both checkpoints).
rollout_dataset = RolloutDataset(
    accessor=accessor,
    eval_time_indices=eval_idx,
    rollout_steps=CONFIG['rollout_steps'],
    mean=mean,
    std=std,
    history_steps=CONFIG['history_steps'],
    max_samples=CONFIG['max_samples'],
)

rollout_loader = DataLoader(
    rollout_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=(device.type == 'cuda'),
    persistent_workers=(CONFIG['num_workers'] > 0),
)

print('Rollout samples:', len(rollout_dataset))

In [ ]:
# Evaluate both checkpoints.
results: Dict[str, Dict[str, np.ndarray]] = {}

for name, bundle in ckpt_bundle.items():
    print('\n' + '=' * 80)
    print(f'Evaluating checkpoint: {name}')
    ckpt = bundle['checkpoint']

    model = build_model_from_checkpoint(
        ckpt=ckpt,
        channels=accessor.channels,
        spatial_shape=accessor.spatial_shape,
    ).to(device)

    rmse, acc = evaluate_rollout(
        model=model,
        loader=rollout_loader,
        mean=mean,
        std=std,
        climatology=climatology,
        lat_w=lat_w,
        device=device,
        amp=CONFIG['amp'],
    )

    results[name] = {'rmse': rmse, 'acc': acc}

    # Free GPU memory before next model.
    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()

print('\nFinished evaluation for all checkpoints.')

lead_steps = np.arange(1, CONFIG['rollout_steps'] + 1)
lead_days = lead_steps * 6.0 / 24.0
var_names = list(accessor.var_names)

# Create long-form dataframe for plotting and analysis.
records = []
for ck_name, mats in results.items():
    rmse_mat = mats['rmse']
    acc_mat = mats['acc']
    for si, step in enumerate(lead_steps):
        for vi, var_name in enumerate(var_names):
            records.append(
                {
                    'checkpoint': ck_name,
                    'lead_step': int(step),
                    'lead_day': float(lead_days[si]),
                    'variable': var_name,
                    'rmse': float(rmse_mat[si, vi]),
                    'acc': float(acc_mat[si, vi]),
                }
            )

metrics_long_df = pd.DataFrame.from_records(records)
display(metrics_long_df.head())
print('metrics_long_df shape:', metrics_long_df.shape)

## 6) Result Visualization

This section creates self-explanatory plots:
- Mean RMSE/ACC vs lead day
- Per-variable RMSE and ACC deltas
- Heatmaps of metric differences over lead time and variable
- Day-5, Day-10, Day-15 comparison bars

In [ ]:
# Mean curves over lead time.
mean_over_vars = (
    metrics_long_df.groupby(['checkpoint', 'lead_step', 'lead_day'], as_index=False)[['rmse', 'acc']]
    .mean()
    .sort_values(['checkpoint', 'lead_step'])
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)

sns.lineplot(data=mean_over_vars, x='lead_day', y='rmse', hue='checkpoint', marker='o', ax=axes[0])
axes[0].set_title('Mean RMSE vs Lead Day (lower is better)')
axes[0].set_xlabel('Lead day')
axes[0].set_ylabel('RMSE')

sns.lineplot(data=mean_over_vars, x='lead_day', y='acc', hue='checkpoint', marker='o', ax=axes[1])
axes[1].set_title('Mean ACC vs Lead Day (higher is better)')
axes[1].set_xlabel('Lead day')
axes[1].set_ylabel('ACC')

plt.show()

# Build pivoted matrices for deltas: emb_1536 - emb_768
rmse_768 = results['emb_768']['rmse']
rmse_1536 = results['emb_1536']['rmse']
acc_768 = results['emb_768']['acc']
acc_1536 = results['emb_1536']['acc']

rmse_delta = rmse_1536 - rmse_768
acc_delta = acc_1536 - acc_768

fig, axes = plt.subplots(1, 2, figsize=(18, max(6, len(var_names) * 0.28)), constrained_layout=True)

sns.heatmap(
    rmse_delta.T,
    cmap='coolwarm',
    center=0.0,
    ax=axes[0],
    cbar_kws={'label': 'RMSE delta (emb_1536 - emb_768)'},
)
axes[0].set_title('Per-variable RMSE delta over lead steps\n(negative means emb_1536 is better)')
axes[0].set_xlabel('Lead step index')
axes[0].set_ylabel('Variable index')

sns.heatmap(
    acc_delta.T,
    cmap='coolwarm',
    center=0.0,
    ax=axes[1],
    cbar_kws={'label': 'ACC delta (emb_1536 - emb_768)'},
)
axes[1].set_title('Per-variable ACC delta over lead steps\n(positive means emb_1536 is better)')
axes[1].set_xlabel('Lead step index')
axes[1].set_ylabel('Variable index')

plt.show()

In [ ]:
# Per-variable average metrics and deltas.
per_var = (
    metrics_long_df.groupby(['checkpoint', 'variable'], as_index=False)[['rmse', 'acc']]
    .mean()
)

per_var_wide = (
    per_var.pivot(index='variable', columns='checkpoint', values=['rmse', 'acc'])
    .sort_index()
)

per_var_compare = pd.DataFrame(
    {
        'variable': per_var_wide.index,
        'rmse_emb_768': per_var_wide[('rmse', 'emb_768')].values,
        'rmse_emb_1536': per_var_wide[('rmse', 'emb_1536')].values,
        'acc_emb_768': per_var_wide[('acc', 'emb_768')].values,
        'acc_emb_1536': per_var_wide[('acc', 'emb_1536')].values,
    }
)
per_var_compare['rmse_delta_1536_minus_768'] = per_var_compare['rmse_emb_1536'] - per_var_compare['rmse_emb_768']
per_var_compare['acc_delta_1536_minus_768'] = per_var_compare['acc_emb_1536'] - per_var_compare['acc_emb_768']

# Sort for readable plots.
rmse_sorted = per_var_compare.sort_values('rmse_delta_1536_minus_768')
acc_sorted = per_var_compare.sort_values('acc_delta_1536_minus_768', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(20, max(8, len(var_names) * 0.25)), constrained_layout=True)

sns.barplot(data=rmse_sorted, y='variable', x='rmse_delta_1536_minus_768', color='#4575b4', ax=axes[0])
axes[0].axvline(0.0, color='black', linewidth=1)
axes[0].set_title('Variable-wise RMSE delta (emb_1536 - emb_768)')
axes[0].set_xlabel('Delta RMSE (negative is better for emb_1536)')
axes[0].set_ylabel('Variable')

sns.barplot(data=acc_sorted, y='variable', x='acc_delta_1536_minus_768', color='#d73027', ax=axes[1])
axes[1].axvline(0.0, color='black', linewidth=1)
axes[1].set_title('Variable-wise ACC delta (emb_1536 - emb_768)')
axes[1].set_xlabel('Delta ACC (positive is better for emb_1536)')
axes[1].set_ylabel('Variable')

plt.show()

display(per_var_compare.head())

In [ ]:
# Day-specific comparison (Day 5, 10, 15).
day_steps = {5: 20, 10: 40, 15: 60}
rows = []
for day, step in day_steps.items():
    si = min(step, CONFIG['rollout_steps']) - 1
    for vi, var_name in enumerate(var_names):
        r768 = rmse_768[si, vi]
        r1536 = rmse_1536[si, vi]
        a768 = acc_768[si, vi]
        a1536 = acc_1536[si, vi]
        rows.append(
            {
                'day': day,
                'variable': var_name,
                'rmse_emb_768': float(r768),
                'rmse_emb_1536': float(r1536),
                'acc_emb_768': float(a768),
                'acc_emb_1536': float(a1536),
                'rmse_delta_1536_minus_768': float(r1536 - r768),
                'acc_delta_1536_minus_768': float(a1536 - a768),
            }
        )

day_compare_df = pd.DataFrame(rows)

for day in sorted(day_compare_df['day'].unique()):
    part = day_compare_df[day_compare_df['day'] == day].copy()

    fig, axes = plt.subplots(1, 2, figsize=(20, max(8, len(part) * 0.22)), constrained_layout=True)

    part_rmse = part.sort_values('rmse_delta_1536_minus_768')
    sns.barplot(data=part_rmse, y='variable', x='rmse_delta_1536_minus_768', color='#74add1', ax=axes[0])
    axes[0].axvline(0.0, color='black', linewidth=1)
    axes[0].set_title(f'Day {day}: RMSE delta (emb_1536 - emb_768)')
    axes[0].set_xlabel('Delta RMSE (negative means emb_1536 better)')

    part_acc = part.sort_values('acc_delta_1536_minus_768', ascending=False)
    sns.barplot(data=part_acc, y='variable', x='acc_delta_1536_minus_768', color='#f46d43', ax=axes[1])
    axes[1].axvline(0.0, color='black', linewidth=1)
    axes[1].set_title(f'Day {day}: ACC delta (emb_1536 - emb_768)')
    axes[1].set_xlabel('Delta ACC (positive means emb_1536 better)')

    plt.show()